# Bronze ingestion: Auto Loader (JSON files on a volume)

Incremental file ingestion. Auto Loader tracks which files it has already seen
in the checkpoint, so re-running is cheap and idempotent: only new files are read.

`trigger(availableNow=True)` makes this a batch-shaped job that drains whatever
has landed and exits, rather than a always-on stream.

Copy this notebook per source; only the four constants below change.

In [0]:
from pyspark.sql import functions as F

In [0]:
# Per-source constants: the only lines that change when you copy this notebook.
SOURCE_NAME = "events"        # subdirectory under the landing volume
TABLE_NAME = "events_raw"     # target table in the bronze schema
SCHEMA = "bronze"
FILE_FORMAT = "json"

In [0]:
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
CATALOG_PREFIX = configs.get("catalog_prefix", "rearc")
INITIAL_RUN = configs.get("initial_run", "False").lower() == "true"

CATALOG = f"{CATALOG_PREFIX}_{ENV}"
INGEST_CATALOG = f"{CATALOG_PREFIX}_ingest"
LANDING_BASE = f"/Volumes/{INGEST_CATALOG}/{ENV}/landing"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

source_path = f"{LANDING_BASE}/{SOURCE_NAME}"
target_table = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
checkpoint_path = f"{CHECKPOINT_BASE}/{TABLE_NAME}/"

print(f"{source_path} -> {target_table} (checkpoint: {checkpoint_path})")

In [0]:
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", FILE_FORMAT)
    .option("cloudFiles.schemaLocation", checkpoint_path)
    # addNewColumns: a new field in the source fails the run once, then the
    # restart picks up the widened schema. Nothing is silently dropped.
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    # anything that does not fit the inferred schema is preserved here rather
    # than discarded, so bronze stays lossless
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
query = (
    df.writeStream.trigger(availableNow=True)
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .toTable(target_table)
)

query.awaitTermination()
print(f"{target_table}: {spark.table(target_table).count():,} rows")

In [0]:
# One-time table properties. Guarded by a job parameter rather than run every
# time: ALTER TABLE on every ingestion run is wasteful, not a production pattern.
if INITIAL_RUN:
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (_ingested_at)")